# Plankton Community Time Series (2019–2024)
Daily density measurements (counts µL⁻¹) from a lake monitoring programme.
The heatmap shows all 82 taxa simultaneously, with species sorted by their seasonal peak month.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ── Load & clean ──────────────────────────────────────────────────────────────
df = pd.read_csv('TimeSeries_countsuL_clean.csv', index_col=0, parse_dates=True)
df.index.name = 'date'

# Drop rows that are completely empty (first 5 dates have no observations)
df = df.dropna(how='all')

# Fill remaining NaNs with 0 (missing counts = not detected)
df = df.fillna(0)

print(f'Date range : {df.index[0].date()} → {df.index[-1].date()}')
print(f'Taxa       : {df.shape[1]}')
print(f'Daily obs  : {df.shape[0]}')

In [ ]:
# ── Monthly resampling ────────────────────────────────────────────────────────
# Use month-start frequency so x-axis aligns cleanly with years
df_monthly = df.resample('MS').mean()

# ── Log₁₀ transform (ε prevents log(0) = -∞) ─────────────────────────────────
eps = df[df > 0].min().min() * 0.1          # 10× below the smallest nonzero value
df_log = np.log10(df_monthly + eps)

# ── Per-species normalisation to [0, 1] ──────────────────────────────────────
# Lets sparse and dominant taxa share the same colour axis.
df_norm = df_log.copy()
for col in df_norm.columns:
    lo, hi = df_norm[col].min(), df_norm[col].max()
    if hi > lo:
        df_norm[col] = (df_norm[col] - lo) / (hi - lo)
    else:
        df_norm[col] = 0.0

# ── Sort taxa by their mean seasonal peak month ───────────────────────────────
seasonal = df.groupby(df.index.month).mean()          # shape (12, n_taxa)
peak_month = seasonal.idxmax(axis=0)                  # month of max for each taxon
sorted_taxa = peak_month.sort_values().index.tolist()

# Matrix: rows = taxa (sorted), columns = monthly time steps
mat = df_norm[sorted_taxa].T.values                   # shape (n_taxa, n_months)

In [ ]:
# ── Figure ────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 11))
fig.patch.set_facecolor('#0f1117')

# --- Grid layout: main heatmap (top) + total biomass strip (bottom) ----------
gs = fig.add_gridspec(2, 2,
                      height_ratios=[11, 2.5],
                      width_ratios=[1, 0.018],
                      hspace=0.08, wspace=0.03)

ax_heat  = fig.add_subplot(gs[0, 0])
ax_cbar  = fig.add_subplot(gs[0, 1])
ax_total = fig.add_subplot(gs[1, 0])

for ax in [ax_heat, ax_cbar, ax_total]:
    ax.set_facecolor('#0f1117')

# ── Heatmap ───────────────────────────────────────────────────────────────────
cmap = plt.get_cmap('inferno')
cmap.set_under('#0f1117')          # zero-density cells blend with background

im = ax_heat.imshow(
    mat,
    aspect='auto',
    origin='upper',
    cmap=cmap,
    vmin=0.05,                      # mask the near-zero background
    vmax=1.0,
    interpolation='nearest'
)

# ── Y-axis: species labels ────────────────────────────────────────────────────
n_taxa = len(sorted_taxa)
ax_heat.set_yticks(range(n_taxa))
ax_heat.set_yticklabels(
    [t.replace('_', ' ') for t in sorted_taxa],
    fontsize=6.5, color='#cccccc'
)
ax_heat.tick_params(axis='y', length=0, pad=4)

# ── X-axis: monthly time labels ───────────────────────────────────────────────
dates = df_monthly.index
n_months = len(dates)

# Place a tick at every January
jan_idx  = [i for i, d in enumerate(dates) if d.month == 1]
jan_labels = [str(dates[i].year) for i in jan_idx]

ax_heat.set_xticks(jan_idx)
ax_heat.set_xticklabels(jan_labels, color='#cccccc', fontsize=9)
ax_heat.tick_params(axis='x', which='both', length=0, labeltop=False, labelbottom=False)

# Vertical year separators
for xi in jan_idx:
    ax_heat.axvline(xi - 0.5, color='#ffffff22', linewidth=0.6, linestyle='--')

# Subtle horizontal grid between species
ax_heat.set_yticks([y - 0.5 for y in range(1, n_taxa)], minor=True)
ax_heat.grid(which='minor', axis='y', color='#ffffff08', linewidth=0.4)

ax_heat.set_ylabel('Taxon  (sorted by seasonal peak)', color='#cccccc', fontsize=10, labelpad=8)
ax_heat.set_title(
    'Lake Plankton Community  ·  2019–2024',
    color='white', fontsize=14, fontweight='bold', pad=12
)
for spine in ax_heat.spines.values():
    spine.set_visible(False)

# ── Colour bar ────────────────────────────────────────────────────────────────
cb = fig.colorbar(im, cax=ax_cbar)
cb.set_label('Relative density\n(normalised log₁₀)',
             color='#cccccc', fontsize=8, labelpad=8)
cb.ax.yaxis.set_tick_params(color='#cccccc', labelsize=7, length=3)
plt.setp(cb.ax.yaxis.get_ticklabels(), color='#cccccc')
cb.outline.set_visible(False)

# ── Bottom strip: total community density ─────────────────────────────────────
total = df_monthly[sorted_taxa].sum(axis=1)           # raw monthly total

ax_total.fill_between(
    range(n_months), total.values,
    color='#e8a838', alpha=0.85, linewidth=0
)
ax_total.plot(range(n_months), total.values,
              color='#f5d06e', linewidth=0.8)

ax_total.set_xlim(-0.5, n_months - 0.5)
ax_total.set_ylim(0)
ax_total.set_xticks(jan_idx)
ax_total.set_xticklabels(jan_labels, color='#cccccc', fontsize=9)
ax_total.tick_params(axis='x', length=0)
ax_total.tick_params(axis='y', labelsize=7, labelcolor='#aaaaaa', length=2, color='#aaaaaa')
ax_total.set_ylabel('Total\ndensity\n(µL⁻¹)', color='#aaaaaa', fontsize=7.5, labelpad=6)
ax_total.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.3f'))

for xi in jan_idx:
    ax_total.axvline(xi - 0.5, color='#ffffff22', linewidth=0.6, linestyle='--')

for spine in ax_total.spines.values():
    spine.set_visible(False)
ax_total.spines['bottom'].set_visible(True)
ax_total.spines['bottom'].set_color('#555555')

# ── Save & show ───────────────────────────────────────────────────────────────
plt.savefig('plankton_heatmap.png', dpi=180, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('Figure saved → plankton_heatmap.png')